# Semantic Axis Exploration

This notebook explores semantic opposition axes in literary text using sentence-transformer embeddings.

A semantic axis is defined by two sets of anchor examples representing opposite poles, such as `light` and `dark`. Each pole is represented by a centroid in embedding space. The axis vector is computed as the difference between the two centroids, and each text segment is projected onto that axis.

This differs from the anchored sentiment scoring procedure used in `02_anchored_sentiment_analysis.ipynb`. In that notebook, the score is computed as the difference between mean similarity to positive and negative anchors. In this notebook, the score is computed by projecting each segment embedding onto a constructed semantic axis.

In [ ]:
import sys

import numpy as np
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

In [ ]:
from literary_nlp.axes import (
    compute_centroid,
    l2_normalise,
    project_score_from_embedding,
    project_score_from_text,
    project_scores_from_texts,
)

## Load Sentence Transformer model

In [ ]:
# Replace this with your actual model path
MODEL_PATH = Path("../models/tolkien_sentence_transformer_epoch_1")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Could not find model at {MODEL_PATH}. "
        "Update MODEL_PATH to your local sentence-transformer model directory."
    )

model = SentenceTransformer(str(MODEL_PATH))

## Preprocessing and segmentation

The sentiment-scoring section expects a dataframe named `segments_df` with at least the following columns:

- `segment_text`: the text segment to be scored;
- `chapter`: the chapter or section label aligned with the segment;
- `segment_type`: optional information about the segment type, such as prose, dialogue, verse, or unknown.

Different source formats require different preprocessing rules. Where possible, structured formats such as TEI/XML or HTML should be preferred because paragraph, chapter, and verse boundaries may already be marked. When only plain text is available, source-specific rules must be checked and documented.

The optional parser cells below illustrate possible preprocessing routes. The default workflow in this notebook uses a prepared plain-text file with explicit chapter markers.

The prepared plain-text parser below is used for the local LOTR file in this project. It assumes chapter markers in the form `###CHAPTER:` and a local paragraph-start convention based on leading spaces. These rules are source-specific.

### TEI/XML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

TEI_PATH = Path("data/source.xml")

def segments_from_tei(path: Path) -> pd.DataFrame:
    """
    Extract prose paragraphs and verse blocks from a TEI/XML file.

    This is a template parser. TEI structures vary, so tag names and
    attributes may need to be adapted for a specific corpus.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "xml")

    rows = []

    for div in soup.find_all("div"):
        chapter_head = div.find("head")
        chapter = chapter_head.get_text(" ", strip=True) if chapter_head else "Unknown"

        for element in div.find_all(["p", "lg"], recursive=True):
            if element.name == "p":
                segment_type = "prose"
                text = element.get_text(" ", strip=True)

            elif element.name == "lg":
                segment_type = "verse"
                lines = [line.get_text(" ", strip=True) for line in element.find_all("l")]
                text = " / ".join(lines) if lines else element.get_text(" ", strip=True)

            else:
                continue

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": chapter,
                        "segment_type": segment_type,
                    }
                )

    return pd.DataFrame(rows)

### HTML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

HTML_PATH = Path("data/source.html")

def segments_from_html(path: Path) -> pd.DataFrame:
    """
    Extract paragraphs from an HTML file.

    This works best for HTML/EPUB-derived texts where paragraphs are
    marked with <p> tags. Chapter detection may need to be adapted
    depending on the source.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    rows = []
    current_chapter = "Unknown"

    for element in soup.find_all(["h1", "h2", "h3", "p"]):
        if element.name in ["h1", "h2", "h3"]:
            current_chapter = element.get_text(" ", strip=True)

        elif element.name == "p":
            text = element.get_text(" ", strip=True)

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": current_chapter,
                        "segment_type": "prose",
                    }
                )

    return pd.DataFrame(rows)

## Prepared plain-text parser used in this notebook

In [ ]:
from literary_nlp.preprocessing import (
    build_segments_df_from_plaintext,
    source_format_diagnostics,
)

In [ ]:
CORPUS_PATH = Path("data/source.txt")

segments_df = build_segments_df_from_plaintext(
    CORPUS_PATH,
    min_tok_narr=80,
    max_tok_narr=300,
    min_tok_dial=60,
    max_tok_dial=180,
    max_dialogue_turns=6,
)

segments_df.head()

## Standard segment dataframe

The segmentation step is standardised into a dataframe named `segments_df`. The later sentiment-scoring cells use this dataframe rather than depending on the specific preprocessing method that produced the segments.

At minimum, `segments_df` contains the segment text and aligned chapter label. A `segment_type` column is included so that future versions can distinguish narration, dialogue, verse, or mixed segments.

In the current plain-text parser, final merged segments are labelled as `mixed_or_unknown`. Future versions may propagate `narration`, `dialogue`, or `verse` labels into the final dataframe.

In [ ]:
diagnostics = source_format_diagnostics(
    CORPUS_PATH,
    segments_df=segments_df,
)

diagnostics

In [ ]:
# Check segment counts by chapter
chapter_segment_counts = (
    segments_df["chapter"]
    .value_counts()
    .sort_index()
)

chapter_segment_counts.head()

## Semantic-axis construction

This section constructs a Light–Dark semantic axis from two sets of anchor sentences.

This differs from the anchored sentiment scoring used in notebook 02. Here, the model first computes a centroid for each semantic pole. The axis is then defined as the direction from the Dark centroid to the Light centroid, and each text segment is projected onto that direction.

## Define Light and Dark anchors

The current light/dark axis uses 13 light anchors and 13 dark anchors. The anchor texts are not included because they contain copyrighted material.

In [ ]:
AXIS_ANCHORS_PATH = Path("data/light_dark_anchors.csv")

if not AXIS_ANCHORS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {AXIS_ANCHORS_PATH}. "
        "This file is not included in the public repository because it may contain copyrighted text. "
        "Create a local CSV with columns: sentence_text;axis_label, where axis_label is 'light' or 'dark'."
    )

axis_anchors_df = pd.read_csv(AXIS_ANCHORS_PATH, sep=";")

required_cols = {"sentence_text", "axis_label"}
missing = required_cols - set(axis_anchors_df.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Present columns: {list(axis_anchors_df.columns)}"
    )

axis_anchors_df["axis_label"] = (
    axis_anchors_df["axis_label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

light_anchors = axis_anchors_df.loc[
    axis_anchors_df["axis_label"] == "light",
    "sentence_text",
].tolist()

dark_anchors = axis_anchors_df.loc[
    axis_anchors_df["axis_label"] == "dark",
    "sentence_text",
].tolist()

if not light_anchors:
    raise ValueError("No light anchors found.")

if not dark_anchors:
    raise ValueError("No dark anchors found.")

print(f"Number of light anchors: {len(light_anchors)}")
print(f"Number of dark anchors: {len(dark_anchors)}")

## Encode anchor sentences

In [ ]:
# Encode semantic-axis anchors

light_embeddings = model.encode(
    light_anchors,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

dark_embeddings = model.encode(
    dark_anchors,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Light embeddings shape:", light_embeddings.shape)
print("Dark embeddings shape:", dark_embeddings.shape)

## Compute Light and Dark centroids

In [ ]:
light_centroid = compute_centroid(light_embeddings)
dark_centroid = compute_centroid(dark_embeddings)

print("Light centroid shape:", light_centroid.shape)
print("Dark centroid shape:", dark_centroid.shape)

The centroid is computed by averaging over the anchor-sentence dimension. This preserves the model's embedding dimension while producing one representative vector for each semantic pole.

## Define the semantic axis

In [ ]:
axis_midpoint = (light_centroid + dark_centroid) / 2
light_dark_axis = l2_normalise(light_centroid - dark_centroid)

print("Axis midpoint shape:", axis_midpoint.shape)
print("Light–Dark axis shape:", light_dark_axis.shape)

The axis is oriented so that positive values indicate movement toward the Light pole and negative values indicate movement toward the Dark pole.

## Text Segment Scoring

text → embedding → centre around midpoint → project onto Light–Dark axis → score

In [ ]:
segments_df["light_dark_score"] = project_scores_from_texts(
    texts=segments_df["segment_text"].tolist(),
    model=model,
    midpoint=axis_midpoint,
    axis_unit=light_dark_axis,
    batch_size=64,
)

In [ ]:
# Inspect strongest Light and Dark segments

display_cols = [
    "chapter",
    "token_count",
    "light_dark_score",
    "segment_text",
]

most_light = (
    segments_df
    .sort_values("light_dark_score", ascending=False)
    [display_cols]
    .head(10)
)

most_dark = (
    segments_df
    .sort_values("light_dark_score", ascending=True)
    [display_cols]
    .head(10)
)

most_light, most_dark

## Sanity-check anchor scores

In [ ]:
light_scores = [
    project_score_from_embedding(e, axis_midpoint, light_dark_axis)
    for e in light_embeddings
]

dark_scores = [
    project_score_from_embedding(e, axis_midpoint, light_dark_axis)
    for e in dark_embeddings
]

anchor_df = pd.DataFrame(
    {
        "set": ["LIGHT"] * len(light_anchors) + ["DARK"] * len(dark_anchors),
        "score": light_scores + dark_scores,
    }
)

anchor_df.sort_values("score", ascending=False).reset_index(drop=True)

In [ ]:
print("Mean Light anchor score:", np.mean(light_scores))
print("Mean Dark anchor score:", np.mean(dark_scores))
print("Min Light anchor score:", np.min(light_scores))
print("Max Dark anchor score:", np.max(dark_scores))

## Check internal coherence of anchor sets

In [ ]:
light_sim = cosine_similarity(light_embeddings)
dark_sim = cosine_similarity(dark_embeddings)

def mean_off_diagonal(sim_matrix: np.ndarray) -> float:
    n = sim_matrix.shape[0]
    return (sim_matrix.sum() - np.trace(sim_matrix)) / (n * (n - 1))

print("Mean Light intra-set similarity:", mean_off_diagonal(light_sim))
print("Mean Dark intra-set similarity:", mean_off_diagonal(dark_sim))
print(
    "Light vs Dark centroid cosine similarity:",
    cosine_similarity([light_centroid], [dark_centroid])[0, 0],
)

## Detect possible anchor outliers

In [ ]:
def centroid_similarity_scores(embeddings: np.ndarray, centroid: np.ndarray) -> np.ndarray:
    return cosine_similarity(embeddings, centroid.reshape(1, -1)).flatten()

light_to_light_centroid = centroid_similarity_scores(
    light_embeddings,
    light_centroid,
)

dark_to_dark_centroid = centroid_similarity_scores(
    dark_embeddings,
    dark_centroid,
)
light_outlier_df = pd.DataFrame(
    {
        "anchor_id": range(len(light_anchors)),
        "axis_label": "light",
        "similarity_to_light_centroid": light_to_light_centroid,
    }
).sort_values("similarity_to_light_centroid")

dark_outlier_df = pd.DataFrame(
    {
        "anchor_id": range(len(dark_anchors)),
        "axis_label": "dark",
        "similarity_to_dark_centroid": dark_to_dark_centroid,
    }
).sort_values("similarity_to_dark_centroid")

print("Possible Light outliers:")
display(light_outlier_df.head())

print("\nPossible Dark outliers:")
display(dark_outlier_df.head())

# Visualisation

In [ ]:
from literary_nlp.plotting import add_rolling_score, plot_trajectory

In [ ]:
WINDOW = 20

plot_df = add_rolling_score(
    segments_df,
    score_col="light_dark_score",
    output_col="smoothed_light_dark_score",
    window=WINDOW,
)

plot_df.head()

In [ ]:
fig, ax = plot_trajectory(
    plot_df,
    score_col="smoothed_light_dark_score",
    chapter_col="chapter",
    title=f"Light–Dark semantic-axis trajectory (rolling window = {WINDOW})",
    ylabel="Light (+) ↔ Dark (−)",
    positive_label="Light side",
    negative_label="Dark side",
    positive_color="goldenrod",
    negative_color="midnightblue",
)